# 序列生成问题CNN-LSTM案例的展示：Image Captioning

本实例代码来源于网络上代码的东拼西凑，请谨慎使用！

## 1. 问题简介

图像描述生成（Image Caption）可以理解为将一副图片翻译为一段描述文字，是一个融合计算机视觉与自然语言处理的综合问题。

![](imgs/image_captioning-example.png)

In [15]:
import torch
import torchvision
import torch.nn as nn
import torchvision.models as models

In [16]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
# when batch_size=256, it costs around 5GB GPU memory. In this case, it could be run with 1060ti.

## 2. Image Caption 模型

In [17]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        modules = list(resnet.children())[:-1]   # delete the last fc layer. 
        self.resnet = nn.Sequential(*modules)    # 常规做法，因为我们只要feature
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)  # 匹配LSTM
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)
        
    def forward(self, images):
        with torch.no_grad():   # 不训练resnet50,因此可以不必为梯度分配额外空间！
            features = self.resnet(images)
        features = features.reshape(features.size(0), -1)
        features = self.bn(self.linear(features))
        return features

![Image Captioning 模型结构](imgs/image_captioning.png)

In [18]:
from torch.nn.utils.rnn import pack_padded_sequence

In [19]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers, max_seq_length=20):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.max_seg_length = max_seq_length
        
    def forward(self, features, captions, lengths):
        embeddings = self.embed(captions)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), 1)
        packed = pack_padded_sequence(embeddings, lengths, batch_first=True) 
        hiddens, _ = self.lstm(packed)
        outputs = self.linear(hiddens[0])
        return outputs
    
    def sample(self, features, states=None):
        sampled_ids = []
        inputs = features.unsqueeze(1)
        for i in range(self.max_seg_length):
            hiddens, states = self.lstm(inputs, states)  # hiddens: (batch_size, 1, hidden_size)
            outputs = self.linear(hiddens.squeeze(1))    # outputs:  (batch_size, vocab_size)
            _, predicted = outputs.max(1)                # predicted: (batch_size)
            sampled_ids.append(predicted)
            inputs = self.embed(predicted)               # inputs: (batch_size, embed_size)
            inputs = inputs.unsqueeze(1)                 # inputs: (batch_size, 1, embed_size)
        sampled_ids = torch.stack(sampled_ids, 1)        # sampled_ids: (batch_size, max_seq_length)
        return sampled_ids

### 关于压缩填充张量(pack_padded_sequence)和解压(pad_packed_sequence)的一些说明

在编码文本句向量的时候常常用到，其反函数pad_packed_sequence是更容易理解的：![](imgs/pad_packing_sequence.png)

但是这样的pad方式产生了过多的冗余信息，pack_padded_sequence负责对它进行“压缩”，即pack的含义。和稀疏矩阵按列压缩存储格式类似，添加一个数组指出上述数据每一列的非空白数量即可，如：

batch_sizes = [3,2,2,1,1]

这样，就可以不用存储那些0表示的空白！一个例子：

In [10]:
tensor_in = torch.FloatTensor([[1,2,3,4,5],[6,7,8,0,0],[9,0,0,0,0]]).resize_(3,5,1)

稀疏矩阵的压缩(pack)存储，不仅节省空间，也可以在后续矩阵乘法中节省计算量

In [11]:
pack = nn.utils.rnn.pack_padded_sequence(tensor_in, [5,3,1], batch_first=True)
print('packed:', pack)

packed: PackedSequence(data=tensor([[1.],
        [6.],
        [9.],
        [2.],
        [7.],
        [3.],
        [8.],
        [4.],
        [5.]]), batch_sizes=tensor([3, 2, 2, 1, 1]), sorted_indices=None, unsorted_indices=None)


声明一个RNN用于实施推理计算

In [14]:
batch_size = 3
hidden_size = 3
max_length = 5
n_layers = 1
rnn = nn.RNN(1, hidden_size, n_layers, batch_first=True)
h0 = torch.randn(n_layers, batch_size, hidden_size)

In [15]:
out, _ = rnn(pack, h0)
print('out:', out)

out: PackedSequence(data=tensor([[ 0.6732, -0.7908, -0.1450],
        [ 0.8744, -0.9296, -0.9930],
        [ 0.9344, -0.8977, -0.9993],
        [ 0.7512, -0.8635, -0.6293],
        [ 0.9551, -0.8358, -0.9988],
        [ 0.8533, -0.8217, -0.9086],
        [ 0.9639, -0.8574, -0.9996],
        [ 0.9008, -0.8091, -0.9769],
        [ 0.9246, -0.8192, -0.9922]], grad_fn=<CatBackward0>), batch_sizes=tensor([3, 2, 2, 1, 1]), sorted_indices=None, unsorted_indices=None)


稀疏格式存储能提高计算效率，计算完成之后，解包(pad)成更便于理解的格式（注意解包之后的0行）：

In [16]:
unpacked = nn.utils.rnn.pad_packed_sequence(out)
print('unpacked:', unpacked)

unpacked: (tensor([[[ 0.6732, -0.7908, -0.1450],
         [ 0.8744, -0.9296, -0.9930],
         [ 0.9344, -0.8977, -0.9993]],

        [[ 0.7512, -0.8635, -0.6293],
         [ 0.9551, -0.8358, -0.9988],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.8533, -0.8217, -0.9086],
         [ 0.9639, -0.8574, -0.9996],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.9008, -0.8091, -0.9769],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.9246, -0.8192, -0.9922],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<CopySlices>), tensor([5, 3, 1]))


## 3. 数据

### 3.1 Dataset: coco 2017 image caption 

In [2]:
import nltk
import pickle
import argparse
import numpy as np
from PIL import Image
from collections import Counter
import torch.utils.data as data
from pycocotools.coco import COCO

In [3]:
image_path_coco = './dataset/coco2017/images/train2017'            # 图像数据
caption_file_coco = './dataset/coco2017/annotations/captions_train2017.json'   # 标题数据
vocab_coco = './vocab_coco.pkl'                            # 字典

#### 3.1.1 预处理

In [4]:
class Vocabulary(object):
    def __init__(self):
        # 重要的是这两个映射关系(字符的向量化)：
        self.word2idx = {}   # 从字搜序号
        self.idx2word = {}   # 用序号找到字
        self.idx = 0  #用任意一种列表结构均可

    def add_word(self, word):
        if not word in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if not word in self.word2idx:
            return self.word2idx['<unk>']
        return self.word2idx[word]

    def __len__(self):
        return len(self.word2idx)

In [5]:
def build_vocab(json, threshold):
    """Build a simple vocabulary wrapper."""
    coco = COCO(json)
    counter = Counter()
    ids = coco.anns.keys()
    for i, id in enumerate(ids):
        caption = str(coco.anns[id]['caption'])
        tokens = nltk.tokenize.word_tokenize(caption.lower())
        counter.update(tokens)

        if (i+1) % 1000 == 0:
            print("[{}/{}] Tokenized the captions.".format(i+1, len(ids)))

    # If the word frequency is less than 'threshold', then the word is discarded.
    words = [word for word, cnt in counter.items() if cnt >= threshold]

    # Create a vocab wrapper and add some special tokens.
    vocab = Vocabulary()
    vocab.add_word('<pad>')  # 可以认为是“系统关键字”
    vocab.add_word('<start>')
    vocab.add_word('<end>')
    vocab.add_word('<unk>')

    # Add the words to the vocabulary.
    for i, word in enumerate(words):
        vocab.add_word(word)
    return vocab

如果时首次运行，须先建立和caption相关的字典，以便于构建词向量。

In [10]:
# nltk.download('punkt')   # nltk第一次使用前，需要下载一些必要的数据文件

In [11]:
#vocab = build_vocab(json=caption_file_coco, threshold=4)

In [12]:
#with open(vocab_coco, 'wb') as f:
#        pickle.dump(vocab, f)
#print("Saved the vocabulary wrapper to '{}'".format(vocab_coco))

In [6]:
with open(vocab_coco, 'rb') as f:
        vocab = pickle.load(f)

In [7]:
print("Total vocabulary size: {}".format(len(vocab)))

Total vocabulary size: 11544


#### 3.1.2 定义数据集

In [8]:
class CocoDataset(data.Dataset):
    def __init__(self, root, json, vocab, transform=None):
        self.root = root
        self.coco = COCO(json)
        self.ids = list(self.coco.anns.keys())
        self.vocab = vocab
        self.transform = transform

    def __getitem__(self, index):
        coco = self.coco
        vocab = self.vocab
        ann_id = self.ids[index]
        caption = coco.anns[ann_id]['caption']
        img_id = coco.anns[ann_id]['image_id']
        
        path = coco.loadImgs(img_id)[0]['file_name']
        image = Image.open(os.path.join(self.root, path)).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        # Convert caption (string) to word ids.
        tokens = nltk.tokenize.word_tokenize(str(caption).lower())
        caption = []
        caption.append(vocab('<start>')) # 系统关键字，提示LSTM用
        caption.extend([vocab(token) for token in tokens]) # 向量化文本
        caption.append(vocab('<end>'))   # 系统关键字，提示LSTM用
        target = torch.Tensor(caption)
        
        return image, target    # 返回“图像-文本”对

    def __len__(self):
        return len(self.ids)

In [9]:
def collate_fn(data):
    """Creates mini-batch tensors from the list of tuples (image, caption).
    
    We should build custom collate_fn rather than using default collate_fn, 
    because merging caption (including padding) is not supported in default.

    Args:
        data: list of tuple (image, caption). 
            - image: torch tensor of shape (3, 256, 256).
            - caption: torch tensor of shape (?); variable length.

    Returns:
        images: torch tensor of shape (batch_size, 3, 256, 256).
        targets: torch tensor of shape (batch_size, padded_length).
        lengths: list; valid length for each padded caption.
    """
    # Sort a data list by caption length (descending order).
    data.sort(key=lambda x: len(x[1]), reverse=True)
    images, captions = zip(*data)

    # Merge images (from tuple of 3D tensor to 4D tensor).
    images = torch.stack(images, 0)

    # Merge captions (from tuple of 1D tensor to 2D tensor).
    lengths = [len(cap) for cap in captions]
    targets = torch.zeros(len(captions), max(lengths)).long()
    for i, cap in enumerate(captions):
        end = lengths[i]
        targets[i, :end] = cap[:end]        
    return images, targets, lengths

In [10]:
import torchvision.transforms as transforms

In [31]:
crop_size = 224   # resnet默认的尺寸，这个最好不要变，除非你有合适的CNN特征提取器
batch_size = 256    # 可以把batch_size 调小一些，以节省训练时占用的内存/显存
num_workers = 4     # 1或2也可，没太大影响

In [32]:
trans = transforms.Compose([ 
        transforms.Resize([crop_size,crop_size]),    # 
        transforms.RandomHorizontalFlip(), 
        transforms.ToTensor(), 
        transforms.Normalize((0.485, 0.456, 0.406), 
                             (0.229, 0.224, 0.225))])

万事具备，可以声明训练用的数据源了：

In [33]:
coco = CocoDataset(root=image_path_coco, json=caption_file_coco, vocab=vocab, transform=trans)

loading annotations into memory...
Done (t=1.19s)
creating index...
index created!


In [34]:
data_loader = torch.utils.data.DataLoader(dataset=coco, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)

### 3.2 Dataset : ai_challenger

实验的数据来自于[AI Challenger图像描述](https://challenger.ai/competition/caption/),对应的训练数据为(ai_challenger_caption_train_20170902.zip)。可以从[这里下载](http://pytorch-1252820389.cosbj.myqcloud.com/caption.pth)处理好的caption数据,对应的image数据太大，须自行下载

In [ ]:
# 原始图片和caption的json文件
img_path = '/datasets/ai_challenger/caption_train_images_20170902'   
captions = '/datasets/ai_challenger/caption_train_annotations_20170902.json'

 TODO...

## 4. 训练

In [21]:
from torchnet import meter
import tqdm
import os

In [42]:
embed_size = 256
hidden_size = 512
num_layers = 6

In [43]:
encoder = EncoderCNN(embed_size).to(device)

In [44]:
# encoder.eval()

In [45]:
decoder = DecoderRNN(embed_size, hidden_size, len(vocab) , num_layers).to(device) 

In [26]:
# decoder.eval()

In [46]:
# Loss and optimizer
learning_rate = 0.0005
criterion = nn.CrossEntropyLoss()
params = list(decoder.parameters()) + list(encoder.linear.parameters()) + list(encoder.bn.parameters())
optimizer = torch.optim.Adam(params, lr=learning_rate)

In [39]:
model_path = 'checkpoints'  # 模型保存前缀
total_step = len(data_loader)
num_epochs = 10
log_step = 200
save_step = 1000

In [40]:
total_step

2312

In [47]:
for epoch in range(num_epochs):
    for i, (images, captions, lengths) in enumerate(data_loader):
            
        # Set mini-batch dataset
        images = images.to(device)
        captions = captions.to(device)
        targets = pack_padded_sequence(captions, lengths, batch_first=True)[0]
            
        # Forward, backward and optimize
        features = encoder(images)
        outputs = decoder(features, captions, lengths)
        loss = criterion(outputs, targets)
        decoder.zero_grad()
        encoder.zero_grad()
        loss.backward()
        optimizer.step()

        # Print log info
        if (i+1) % log_step == 0:
            print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Perplexity: {:5.4f}'
                    .format(epoch+1, num_epochs, i+1, total_step, loss.item(), np.exp(loss.item()))) 
                
        # Save the model checkpoints
        if (i+1) % save_step == 0:
            torch.save(decoder.state_dict(), os.path.join(model_path, 'decoder-{}-{}.ckpt'.format(epoch+1, i+1)))
            torch.save(encoder.state_dict(), os.path.join(model_path, 'encoder-{}-{}.ckpt'.format(epoch+1, i+1)))

Epoch [0/10], Step [200/2312], Loss: 4.6121, Perplexity: 100.6937
Epoch [0/10], Step [400/2312], Loss: 4.5055, Perplexity: 90.5117
Epoch [0/10], Step [600/2312], Loss: 4.4260, Perplexity: 83.5978
Epoch [0/10], Step [800/2312], Loss: 4.4313, Perplexity: 84.0408
Epoch [0/10], Step [1000/2312], Loss: 4.5242, Perplexity: 92.2184
Epoch [0/10], Step [1200/2312], Loss: 4.4740, Perplexity: 87.7093
Epoch [0/10], Step [1400/2312], Loss: 4.4219, Perplexity: 83.2536
Epoch [0/10], Step [1600/2312], Loss: 4.4952, Perplexity: 89.5867
Epoch [0/10], Step [1800/2312], Loss: 4.4708, Perplexity: 87.4305
Epoch [0/10], Step [2000/2312], Loss: 4.4314, Perplexity: 84.0457
Epoch [0/10], Step [2200/2312], Loss: 4.5018, Perplexity: 90.1754
Epoch [1/10], Step [200/2312], Loss: 4.5235, Perplexity: 92.1598
Epoch [1/10], Step [400/2312], Loss: 4.4532, Perplexity: 85.9010
Epoch [1/10], Step [600/2312], Loss: 4.5045, Perplexity: 90.4215
Epoch [1/10], Step [800/2312], Loss: 4.5225, Perplexity: 92.0620
Epoch [1/10], Ste

## 5. 推理

In [ ]:
test_img = 'imgs/example.jpeg'

In [ ]:
# 进一步测试代码，请参考demo_ic_inference.ipynb

## 6. Attentation based 方案

In [17]:
# 进一步参考开源代码: https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning
# 这里，我们用该代码model.py中的encoder和decoder, 数据集则复用前面已经做好的 CocoDataset

In [22]:
import torch.optim
import torch.utils.data
import torchvision.models as models
import torchvision.transforms as transforms

from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence

In [23]:
# from ImageCaptioningTrans.models import Encoder, DecoderWithAttention

In [24]:
class Encoder(nn.Module):
    def __init__(self, encoded_image_size=14):
        super(Encoder, self).__init__()
        self.enc_image_size = encoded_image_size

  #      resnet = models.resnet101(pretrained=True)  # pretrained ImageNet ResNet-101
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove linear and pool layers (since we're not doing classification)
        modules = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)

        # Resize image to fixed size to allow input images of variable size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))

        self.fine_tune()

    def forward(self, images):
        out = self.resnet(images)  # (batch_size, 2048, image_size/32, image_size/32)
        out = self.adaptive_pool(out)  # (batch_size, 2048, encoded_image_size, encoded_image_size)
        out = out.permute(0, 2, 3, 1)  # (batch_size, encoded_image_size, encoded_image_size, 2048)
        return out

    def fine_tune(self, fine_tune=True):
        for p in self.resnet.parameters():
            p.requires_grad = False
        # If fine-tuning, only fine-tune convolutional blocks 2 through 4
        for c in list(self.resnet.children())[5:]:
            for p in c.parameters():
                p.requires_grad = fine_tune

In [25]:
class Attention(nn.Module):

    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)  # linear layer to transform encoded image
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)  # linear layer to transform decoder's output
        self.full_att = nn.Linear(attention_dim, 1)  # linear layer to calculate values to be softmax-ed
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)  # softmax layer to calculate weights

    def forward(self, encoder_out, decoder_hidden):
        att1 = self.encoder_att(encoder_out)  # (batch_size, num_pixels, attention_dim)
        att2 = self.decoder_att(decoder_hidden)  # (batch_size, attention_dim)
        att = self.full_att(self.relu(att1 + att2.unsqueeze(1))).squeeze(2)  # (batch_size, num_pixels)
        alpha = self.softmax(att)  # (batch_size, num_pixels)
        attention_weighted_encoding = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)  # (batch_size, encoder_dim)

        return attention_weighted_encoding, alpha

In [67]:
class DecoderWithAttention(nn.Module):
    
    def __init__(self, attention_dim, embed_dim, decoder_dim, vocab_size, encoder_dim=2048, dropout=0.5):
        super(DecoderWithAttention, self).__init__()

        self.encoder_dim = encoder_dim
        self.attention_dim = attention_dim
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size
        self.dropout = dropout

        self.attention = Attention(encoder_dim, decoder_dim, attention_dim)  # attention network

        self.embedding = nn.Embedding(vocab_size, embed_dim)  # embedding layer
        self.dropout = nn.Dropout(p=self.dropout)
        self.decode_step = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim, bias=True)  # decoding LSTMCell
        self.init_h = nn.Linear(encoder_dim, decoder_dim)  # linear layer to find initial hidden state of LSTMCell
        self.init_c = nn.Linear(encoder_dim, decoder_dim)  # linear layer to find initial cell state of LSTMCell
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)  # linear layer to create a sigmoid-activated gate
        self.sigmoid = nn.Sigmoid()
        self.fc = nn.Linear(decoder_dim, vocab_size)  # linear layer to find scores over vocabulary
        self.init_weights()  # initialize some layers with the uniform distribution

    def init_weights(self):
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def load_pretrained_embeddings(self, embeddings):
        self.embedding.weight = nn.Parameter(embeddings)

    def fine_tune_embeddings(self, fine_tune=True):
        for p in self.embedding.parameters():
            p.requires_grad = fine_tune

    def init_hidden_state(self, encoder_out):
        mean_encoder_out = encoder_out.mean(dim=1)
        h = self.init_h(mean_encoder_out)  # (batch_size, decoder_dim)
        c = self.init_c(mean_encoder_out)
        return h, c

    def forward(self, encoder_out, encoded_captions, caption_lengths):
        """
        :param encoder_out: encoded images, a tensor of dimension (batch_size, enc_image_size, enc_image_size, encoder_dim)
        :param encoded_captions: encoded captions, a tensor of dimension (batch_size, max_caption_length)
        :param caption_lengths: caption lengths, a tensor of dimension (batch_size, 1)
        :return: scores for vocabulary, sorted encoded captions, decode lengths, weights, sort indices
        """

        batch_size = encoder_out.size(0)
        encoder_dim = encoder_out.size(-1)
        vocab_size = self.vocab_size

        # Flatten image
        encoder_out = encoder_out.view(batch_size, -1, encoder_dim)  # (batch_size, num_pixels, encoder_dim)
        num_pixels = encoder_out.size(1)

        # Sort input data by decreasing lengths; why? apparent below
        caption_lengths, sort_ind = caption_lengths.squeeze(1).sort(dim=0, descending=True)
        encoder_out = encoder_out[sort_ind]
        encoded_captions = encoded_captions[sort_ind]

        # Embedding
        embeddings = self.embedding(encoded_captions)  # (batch_size, max_caption_length, embed_dim)

        # Initialize LSTM state
        h, c = self.init_hidden_state(encoder_out)  # (batch_size, decoder_dim)

        # We won't decode at the <end> position, since we've finished generating as soon as we generate <end>
        # So, decoding lengths are actual lengths - 1
        decode_lengths = (caption_lengths - 1).tolist()

        # Create tensors to hold word predicion scores and alphas
        predictions = torch.zeros(batch_size, max(decode_lengths), vocab_size).to(device)
        alphas = torch.zeros(batch_size, max(decode_lengths), num_pixels).to(device)

        # At each time-step, decode by
        # attention-weighing the encoder's output based on the decoder's previous hidden state output
        # then generate a new word in the decoder with the previous word and the attention weighted encoding
        for t in range(max(decode_lengths)):
            batch_size_t = sum([l > t for l in decode_lengths])
            attention_weighted_encoding, alpha = self.attention(encoder_out[:batch_size_t],
                                                                h[:batch_size_t])
            gate = self.sigmoid(self.f_beta(h[:batch_size_t]))  # gating scalar, (batch_size_t, encoder_dim)
            attention_weighted_encoding = gate * attention_weighted_encoding
            h, c = self.decode_step(
                torch.cat([embeddings[:batch_size_t, t, :], attention_weighted_encoding], dim=1),
                (h[:batch_size_t], c[:batch_size_t]))  # (batch_size_t, decoder_dim)
            preds = self.fc(self.dropout(h))  # (batch_size_t, vocab_size)
            predictions[:batch_size_t, t, :] = preds
            alphas[:batch_size_t, t, :] = alpha

        return predictions, encoded_captions, decode_lengths, alphas, sort_ind

In [58]:
import time
from torch import Tensor

In [27]:
from nltk.translate.bleu_score import corpus_bleu

In [28]:
global best_bleu4, epochs_since_improvement, checkpoint, start_epoch, fine_tune_encoder, data_name, word_map

In [29]:
# Model parameters
emb_dim = 512  # dimension of word embeddings
attention_dim = 512  # dimension of attention linear layers
decoder_dim = 512  # dimension of decoder RNN
dropout = 0.5

In [30]:
decoder = DecoderWithAttention(attention_dim=attention_dim,
                                       embed_dim=emb_dim,
                                       decoder_dim=decoder_dim,
                                       vocab_size=len(vocab),
                                       dropout=dropout)

In [36]:
# Training parameters
workers = 1  # for data-loading; right now, only 1 works with h5py
encoder_lr = 1e-4  # learning rate for encoder if fine-tuning
decoder_lr = 4e-4  # learning rate for decoder
grad_clip = 5.  # clip gradients at an absolute value of
alpha_c = 1.  # regularization parameter for 'doubly stochastic attention', as in the paper
best_bleu4 = 0.  # BLEU-4 score right now
print_freq = 100  # print training/validation stats every __ batches
fine_tune_encoder = False  # fine-tune encoder?

In [32]:
decoder_optimizer = torch.optim.Adam(params=filter(lambda p: p.requires_grad, decoder.parameters()),lr=decoder_lr)

In [33]:
encoder = Encoder()

In [ ]:
encoder.fine_tune(False)

In [37]:
encoder_optimizer = torch.optim.Adam(params=filter(lambda p: p.requires_grad, encoder.parameters()), lr=encoder_lr) if fine_tune_encoder else None

In [41]:
# Move to GPU, if available
decoder = decoder.to(device)
encoder = encoder.to(device)

In [42]:
criterion = nn.CrossEntropyLoss().to(device)

In [50]:
start_epoch = 0
epochs = 10  # number of epochs to train for (if early stopping is not triggered)
epochs_since_improvement = 0  # keeps track of number of epochs since there's been an improvement in validation BLEU

log_step = 50
save_step = 1000   

In [ ]:
decoder.train() 
encoder.train()

In [62]:
test = torch.tensor([1,2,3]).to(device)

In [68]:
for epoch in range(start_epoch, epochs):
    
 #   batch_time = AverageMeter()  # forward prop. + back prop. time
 #   data_time = AverageMeter()  # data loading time
 #   losses = AverageMeter()  # loss (per word decoded)
  #  top5accs = AverageMeter()  # top5 accuracy

    start = time.time()
    # Batches

            
    for i, (imgs, caps, caplens) in enumerate(data_loader):

     #   print('caps = {} with length {}'.format(caps, caplens))
        
        # Move to GPU, if available
        imgs = imgs.to(device)
        caps = caps.to(device)
        caplens = torch.tensor([caplens]).to(device)

        # Forward prop.
        imgs = encoder(imgs)
        scores, caps_sorted, decode_lengths, alphas, sort_ind = decoder(imgs, caps, caplens)

        # Since we decoded starting with <start>, the targets are all words after <start>, up to <end>
        targets = caps_sorted[:, 1:]

        # Remove timesteps that we didn't decode at, or are pads
        # pack_padded_sequence is an easy trick to do this
        scores, _ = pack_padded_sequence(scores, decode_lengths, batch_first=True)
        targets, _ = pack_padded_sequence(targets, decode_lengths, batch_first=True)

        # Calculate loss
        loss = criterion(scores, targets)

        # Add doubly stochastic attention regularization
        loss += alpha_c * ((1. - alphas.sum(dim=1)) ** 2).mean()

        # Back prop.
        decoder_optimizer.zero_grad()
        if encoder_optimizer is not None:
            encoder_optimizer.zero_grad()
        loss.backward()

        # Clip gradients
        if grad_clip is not None:
            clip_gradient(decoder_optimizer, grad_clip)
            if encoder_optimizer is not None:
                clip_gradient(encoder_optimizer, grad_clip)

        # Update weights
        decoder_optimizer.step()
        if encoder_optimizer is not None:
            encoder_optimizer.step()

        # Keep track of metrics
        top5 = accuracy(scores, targets, 5)
        losses = loss.item()
        top5accs = top5
        batch_time = time.time() - start

        start = time.time()
      
                
        # Save the model checkpoints
        if (i+1) % save_step == 0:
            torch.save(decoder.state_dict(), os.path.join(model_path, 'decoder-{}-{}.ckpt'.format(epoch+1, i+1)))
            torch.save(encoder.state_dict(), os.path.join(model_path, 'encoder-{}-{}.ckpt'.format(epoch+1, i+1)))
            
        # Print status
        if (i+1) % log_step == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Batch Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Top-5 Accuracy {top5.val:.3f} ({top5.avg:.3f})'.format(epoch, i+1, len(train_loader), batch_time=batch_time, loss=losses, top5=top5accs))

TypeError: zeros(): argument 'size' must be tuple of SymInts, but found element of type list at pos 2